# How to Build Agents

An **agent** is a role definition for an AI model: what it is, what it knows, what it can do, and when to use it. Agents are composable: an orchestrator agent delegates to specialist agents, which call tools.

This notebook covers how to define, deploy, and compose agents in the Mad House model.

## Agent Types

### Orchestrator
Receives a high-level task, breaks it down, and delegates to specialists. Does not do the work itself.

Example: `@delegator` receives "audit the VPS", decides it needs `vps-audit`, `security-check`, and `cost-report` agents, and coordinates them.

### Specialist
Has deep context on one domain. Does the work. Returns a structured result.

Example: `@security-reviewer` knows the OWASP top 10, has access to the codebase, and returns a list of findings.

### Tool-calling agent
Wraps a set of tools into a callable unit. The simplest form: a skill with a single script.

Example: `coolify-deploy` wraps 5 scripts (list, search, start, status, logs) and orchestrates them for the user.

## Agent File Format

Agents are Markdown files with YAML frontmatter:

```markdown
---
name: code-reviewer
description: Reviews code for quality, correctness, and security. Use when asking for
  a second opinion on a PR, a diff, or a file. Triggers on: review, audit code, check this.
tools:
  - Read
  - Bash
  - WebSearch
model: claude-sonnet-4-6
---

# Code Reviewer

You are a senior engineer reviewing code. Your job is to surface issues, not fix them
unless asked.

## What to check

1. Correctness: does the code do what the author intended?
2. Security: are there injection risks, improper secret handling, or open attack surfaces?
3. Simplicity: is there a simpler path?
4. Edge cases: what breaks at the boundary?

## Output format

Return a numbered list of findings. For each: location, issue, severity (low/medium/high),
and a one-line fix suggestion.
```

## Agent Frontmatter Reference

| Field | Purpose |
|-------|---------|
| `name` | Unique identifier. Referenced with `@name` in prompts. |
| `description` | Always loaded. Must clearly state when to use this agent. |
| `tools` | Which Claude Code tools this agent can access. |
| `model` | Which model to use. Defaults to session model if omitted. |
| `subagent_type` | For Agent tool calls: maps to a registered agent type. |

Keep descriptions specific. Vague descriptions cause agents to activate at the wrong time or not at all.

## The Delegation Pattern

Orchestrators delegate using the Agent tool. The pattern:

1. Receive a complex task
2. Identify which specialists are needed
3. Dispatch each specialist with a self-contained prompt (they have no context from the conversation)
4. Collect results and synthesize

**Critical rule:** every subagent prompt must be self-contained. The subagent starts cold. It does not know what the user said, what files you have read, or what the conversation history is. Brief it like a colleague who just walked into the room.

```python
# What the orchestrator does (pseudocode)
task = "Audit the chopsticks bot for security issues"

# Dispatch a specialist
Agent(
    description="Security review of chopsticks bot",
    prompt="""
    Review the Discord bot at ~/dev/mad-house/chopsticks for security issues.
    The bot uses postgres and redis. We are looking for:
    1. SQL injection via user input
    2. Exposed secrets in code or config
    3. Unsafe command execution
    Return a numbered list of findings with severity and file:line references.
    """
)
```

## Multi-Agent Workflow

For tasks that can be parallelized, dispatch multiple agents in one message:

```
User: Audit both chopsticks and kinetrak-ai for security issues.

Orchestrator:
  [parallel]
  Agent 1: security review of chopsticks
  Agent 2: security review of kinetrak-ai
  [wait for both]
  Synthesize: combine findings, deduplicate, rank by severity
```

Independent subtasks should always run in parallel. Sequential dispatch is only needed when one agent's output is another agent's input.

### Isolation modes

When a subagent will modify files, use `isolation: "worktree"`. This gives the agent a clean git worktree so it cannot corrupt the main working directory.

```python
Agent(
    description="Implement feature X",
    isolation="worktree",
    prompt="..."
)
```

## Context Management

Context is expensive. The longer a conversation, the more tokens every message costs.

### Rules

1. **Never load what you do not need.** An agent reviewing one file should not read the whole repo.
2. **Use subagents for isolation.** Subagents protect the main context window from search results, large diffs, and noisy outputs.
3. **Session persistence.** Important findings should be written to files, not held in context. The session ends; files do not.
4. **Progressive loading.** Load skill metadata (small) in every session. Load skill body (medium) on activation. Load scripts (large) only when executing.

### The `AGENTS.md` pattern

Every repo in the Mad House org should have an `AGENTS.md` that an agent reads on first contact with that directory. It answers: what is this, what should I not touch, what are the active tasks?

`AGENTS.md` is never committed to public repos. It is in `.gitignore_global`.

## Building Your First Agent

We will build a `changelog-writer` agent that reads git log and writes a CHANGELOG entry.

In [ ]:
changelog_agent = '''\
---
name: changelog-writer
description: Reads recent git commits and writes a CHANGELOG.md entry in Keep a Changelog
  format. Use when releasing a version or when the user says "update the changelog".
tools:
  - Bash
  - Read
  - Edit
---

# Changelog Writer

You write changelog entries. You do not editorialize; you summarize what the commits say.

## Steps

1. Run `git log --oneline --since="last tag"` (or last 30 days if no tags exist).
2. Categorize commits: Added, Changed, Fixed, Removed, Security.
3. Read `CHANGELOG.md`. If it does not exist, create it with the standard Keep a Changelog header.
4. Prepend a new `## [Unreleased]` section with today\'s date.
5. Under each category, write one bullet per logical change (not per commit).

## Rules

- No em dashes.
- Bullet text starts with a verb in past tense: "Added", "Fixed", "Removed".
- Group related commits into one bullet.
- Do not include merge commits or changelog-only commits.
'''

print(changelog_agent)

## Agent vs. Skill: When to Use Which

| Use | Agent | Skill |
|-----|-------|-------|
| Pure instructions, no scripts | Yes | Yes |
| Calls external APIs or shell commands | Use skill's scripts/ | |
| Delegates to other agents | Yes | |
| Specialized domain knowledge | Yes | |
| Wraps a repeatable workflow | | Yes |
| Published to a public registry | | Yes (scrubbed) |

A skill with no scripts is basically an agent instruction set. A skill with scripts is a tool-calling agent. Use whichever model fits the Claude Code invocation path for your use case.